## Scene Cropping: Evaluation of PointNet++ MSG


In [1]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import plotly.graph_objects as go
import plotly.express as px

# Verify
print(f"PyTorch       : {torch.__version__}")
import plotly
print(f"Plotly        : {plotly.__version__}")

# Device (MPS on Mac)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device        : {device}")

# Paths
_cwd = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == "notebooks" else _cwd

DATA_DIR_M40    = os.path.join(PROJECT_ROOT, "data", "ModelNet40")
CACHE_DIR_M40   = os.path.join(PROJECT_ROOT, "data", "cache_modelnet40_1024pts")
CHECKPOINT_DIR  = os.path.join(PROJECT_ROOT, "checkpoints")
S3DIS_DIR       = os.path.join(PROJECT_ROOT, "data", "S3DIS")

# MSG checkpoint
msg_ckpt_path = os.path.join(CHECKPOINT_DIR, "pointnetpp_msg_best.pt")
assert os.path.exists(msg_ckpt_path), f"MSG checkpoint missing at {msg_ckpt_path}"
print(f"\nMSG checkpoint found: {msg_ckpt_path}")
print(f"  Size: {os.path.getsize(msg_ckpt_path)/1024**2:.1f} MB")

# ModelNet40 classes (what our model knows)
CLASSES_M40 = [
    "airplane", "bathtub", "bed", "bench", "bookshelf", "bottle", "bowl",
    "car", "chair", "cone", "cup", "curtain", "desk", "door", "dresser",
    "flower_pot", "glass_box", "guitar", "keyboard", "lamp", "laptop",
    "mantel", "monitor", "night_stand", "person", "piano", "plant", "radio",
    "range_hood", "sink", "sofa", "stairs", "stool", "table", "tent", "toilet",
    "tv_stand", "vase", "wardrobe", "xbox",
]
CLASS_TO_IDX_M40 = {c: i for i, c in enumerate(CLASSES_M40)}
IDX_TO_CLASS_M40 = {i: c for c, i in CLASS_TO_IDX_M40.items()}

print(f"\nReady. ModelNet40 has {len(CLASSES_M40)} classes.")

PyTorch       : 2.12.0
Plotly        : 6.8.0
Device        : mps

MSG checkpoint found: /Users/dosvatsky/3D Object Detection/checkpoints/pointnetpp_msg_best.pt
  Size: 6.8 MB

Ready. ModelNet40 has 40 classes.


## Defining PointNet++ MSG architecture 

In [2]:
# ===== FPS, Ball Query, index_points =====
def farthest_point_sample(xyz, npoint):
    B, N, _ = xyz.shape
    dev = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long, device=dev)
    distance = torch.full((B, N), float("inf"), device=dev)
    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    batch_idx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(npoint):
        centroids[:, i] = farthest
        cxyz = xyz[batch_idx, farthest, :].unsqueeze(1)
        dist = ((xyz - cxyz) ** 2).sum(dim=-1)
        distance = torch.minimum(distance, dist)
        farthest = distance.argmax(dim=-1)
    return centroids


def index_points(points, idx):
    B = points.shape[0]
    vs = list(idx.shape); vs[1:] = [1]*(len(vs)-1)
    rs = list(idx.shape); rs[0] = 1
    bi = torch.arange(B, dtype=torch.long, device=points.device).view(vs).repeat(rs)
    return points[bi, idx, :]


def ball_query(radius, nsample, xyz, new_xyz):
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape
    dev = xyz.device
    gi = torch.arange(N, dtype=torch.long, device=dev).view(1, 1, N).repeat(B, S, 1)
    sd = ((new_xyz.unsqueeze(2) - xyz.unsqueeze(1)) ** 2).sum(dim=-1)
    gi[sd > radius ** 2] = N
    gi = gi.sort(dim=-1)[0][:, :, :nsample]
    gf = gi[:, :, 0:1].repeat(1, 1, nsample); gi[gi == N] = gf[gi == N]
    return gi


class SetAbstractionMSG(nn.Module):
    def __init__(self, npoint, radii, nsamples, in_channel, mlps):
        super().__init__()
        self.npoint, self.radii, self.nsamples = npoint, radii, nsamples
        self.conv_blocks, self.bn_blocks = nn.ModuleList(), nn.ModuleList()
        for mlp in mlps:
            convs, bns = nn.ModuleList(), nn.ModuleList()
            last = in_channel + 3
            for c in mlp:
                convs.append(nn.Conv2d(last, c, 1)); bns.append(nn.BatchNorm2d(c)); last = c
            self.conv_blocks.append(convs); self.bn_blocks.append(bns)

    def forward(self, xyz, features=None):
        fps = farthest_point_sample(xyz, self.npoint)
        new_xyz = index_points(xyz, fps); outs = []
        for i, (r, k) in enumerate(zip(self.radii, self.nsamples)):
            nn_idx = ball_query(r, k, xyz, new_xyz)
            g_xyz = index_points(xyz, nn_idx) - new_xyz.unsqueeze(2)
            if features is not None:
                g = torch.cat([g_xyz, index_points(features, nn_idx)], dim=-1)
            else:
                g = g_xyz
            g = g.permute(0, 3, 1, 2).contiguous()
            for conv, bn in zip(self.conv_blocks[i], self.bn_blocks[i]):
                g = F.relu(bn(conv(g)))
            outs.append(g.max(dim=-1)[0])
        return new_xyz, torch.cat(outs, dim=1).permute(0, 2, 1).contiguous()


class GlobalSetAbstraction(nn.Module):
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.convs, self.bns = nn.ModuleList(), nn.ModuleList()
        last = in_channel
        for c in mlp:
            self.convs.append(nn.Conv1d(last, c, 1)); self.bns.append(nn.BatchNorm1d(c)); last = c

    def forward(self, xyz, features):
        x = torch.cat([xyz, features], dim=-1).permute(0, 2, 1)
        for conv, bn in zip(self.convs, self.bns):
            x = F.relu(bn(conv(x)))
        return x.max(dim=-1)[0]


class PointNetPlusPlusMSG(nn.Module):
    def __init__(self, num_classes=40, dropout=0.5):
        super().__init__()
        self.sa1 = SetAbstractionMSG(512, [0.1, 0.2, 0.4], [16, 32, 128], 0,
                                     [[32, 32, 64], [64, 64, 128], [64, 96, 128]])
        self.sa2 = SetAbstractionMSG(128, [0.2, 0.4, 0.8], [32, 64, 128], 320,
                                     [[64, 64, 128], [128, 128, 256], [128, 128, 256]])
        self.sa_global = GlobalSetAbstraction(640 + 3, [256, 512, 1024])
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,  num_classes),
        )

    def forward(self, xyz):
        l1_xyz, l1_f = self.sa1(xyz, None)
        l2_xyz, l2_f = self.sa2(l1_xyz, l1_f)
        return self.classifier(self.sa_global(l2_xyz, l2_f))


print("PointNet++ MSG architecture defined.")

PointNet++ MSG architecture defined.


## Loading MSG checkpoint and doing sanity check on one ModelNet40 sample

In [3]:
# Load the trained checkpoint
model = PointNetPlusPlusMSG(num_classes=40, dropout=0.5).to(device)
ckpt = torch.load(msg_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"])
model.eval()

print(f"Loaded checkpoint:")
print(f"  Epoch     : {ckpt.get('epoch', '?')}")
print(f"  Val acc   : {ckpt.get('val_acc', '?')}")
print(f"  Weights   : {'EMA' if ckpt.get('is_ema') else 'raw'}")

# Sanity check: run inference on one chair from ModelNet40 cache
# (this confirms the model loads correctly before we feed it real-scene crops)
cache_path = os.path.join(CACHE_DIR_M40, "test.npz")
if os.path.exists(cache_path):
    data = np.load(cache_path)
    points, labels = data["points"], data["labels"]
    chair_idx = int(np.where(labels == CLASS_TO_IDX_M40["chair"])[0][0])
    pts = torch.from_numpy(points[chair_idx]).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(pts)
        pred = logits.argmax(1).item()
        confidence = F.softmax(logits, dim=1).max().item()
    print(f"\nSanity check on a clean ModelNet40 chair:")
    print(f"  True class : chair  (idx {CLASS_TO_IDX_M40['chair']})")
    print(f"  Predicted  : {IDX_TO_CLASS_M40[pred]}  (idx {pred})")
    print(f"  Confidence : {confidence:.3f}")
    print(f"  Correct?   : {pred == CLASS_TO_IDX_M40['chair']}")
else:
    print(f"\nNo ModelNet40 cache at {cache_path} — skipping sanity check.")

Loaded checkpoint:
  Epoch     : 28
  Val acc   : 0.8987034035656402
  Weights   : EMA

Sanity check on a clean ModelNet40 chair:
  True class : chair  (idx 8)
  Predicted  : chair  (idx 8)
  Confidence : 0.936
  Correct?   : True


## S3DIS file parser 

In [1]:
import glob

# S3DIS classes (13 classes in the standard split)
S3DIS_CLASSES = [
    "ceiling", "floor", "wall", "beam", "column", "window", "door",
    "table", "chair", "sofa", "bookcase", "board", "clutter",
]

# Mapping from S3DIS class names to ModelNet40 class names (where they exist)
S3DIS_TO_MODELNET40 = {
    "chair":    "chair",
    "table":    "table",
    "sofa":     "sofa",
    "bookcase": "bookshelf",   # name change
    "door":     "door",
    # ceiling, floor, wall, beam, column, window, board, clutter → no MN40 mapping
}


def load_s3dis_room(room_dir):
    """Load the whole-room point cloud.

    Args:
        room_dir: path to a room folder, e.g. .../Area_5/conferenceRoom_1/

    Returns:
        (N, 6) array of [x, y, z, r, g, b] per point
    """
    room_name = os.path.basename(room_dir.rstrip("/"))
    room_file = os.path.join(room_dir, f"{room_name}.txt")
    assert os.path.exists(room_file), f"Room file not found: {room_file}"
    # np.loadtxt is slow on big files; this is a many-million-point file so we use numpy fromstring fast-path
    data = np.loadtxt(room_file, dtype=np.float32)
    return data  # (N, 6): x, y, z, r, g, b


def load_s3dis_annotations(room_dir):
    """Load each per-object annotation file in the room's Annotations/ folder.

    Returns:
        List of dicts, each with keys:
            'class_name'    : str    (e.g. 'chair', 'table')
            'instance_id'   : int    (chair_1 -> 1, chair_2 -> 2, etc.)
            'points'        : (N, 6) array of [x, y, z, r, g, b]
            'mn40_class'    : str or None — ModelNet40 class if mapped, else None
            'bbox'          : (6,) [x_min, y_min, z_min, x_max, y_max, z_max] axis-aligned
    """
    ann_dir = os.path.join(room_dir, "Annotations")
    assert os.path.exists(ann_dir), f"Annotations folder not found: {ann_dir}"

    objects = []
    for f in sorted(glob.glob(os.path.join(ann_dir, "*.txt"))):
        base = os.path.splitext(os.path.basename(f))[0]   # e.g. "chair_1"
        parts = base.rsplit("_", 1)                       # split on LAST underscore
        if len(parts) != 2:
            continue
        class_name, inst_str = parts
        try:
            inst_id = int(inst_str)
        except ValueError:
            continue
        pts = np.loadtxt(f, dtype=np.float32)
        if pts.ndim == 1:                                 # single-point file (shouldn't happen)
            pts = pts.reshape(1, 6)
        if pts.shape[0] == 0:
            continue
        bbox = np.concatenate([pts[:, :3].min(axis=0), pts[:, :3].max(axis=0)])
        objects.append({
            "class_name": class_name,
            "instance_id": inst_id,
            "points": pts,
            "mn40_class": S3DIS_TO_MODELNET40.get(class_name),
            "bbox": bbox,
        })
    return objects


print(f"Parser functions defined.")
print(f"S3DIS has {len(S3DIS_CLASSES)} classes; {len(S3DIS_TO_MODELNET40)} map directly to ModelNet40.")
print(f"Mapping: {S3DIS_TO_MODELNET40}")

Parser functions defined.
S3DIS has 13 classes; 5 map directly to ModelNet40.
Mapping: {'chair': 'chair', 'table': 'table', 'sofa': 'sofa', 'bookcase': 'bookshelf', 'door': 'door'}


## Listing rooms in Area_5 and picking one

In [5]:
import os

# ============================================================
# Locate S3DIS Area_5
# Your notebook is inside:
# /Users/dosvatsky/3D Object Detection/notebooks
# so we go one level up to reach data/
# ============================================================

PROJECT_ROOT = os.path.abspath("..")
S3DIS_DIR = os.path.join(PROJECT_ROOT, "data", "S3DIS")
AREA_5_DIR = os.path.join(S3DIS_DIR, "Area_5")

assert os.path.exists(S3DIS_DIR), f"S3DIS folder not found: {S3DIS_DIR}"
assert os.path.exists(AREA_5_DIR), f"Area_5 folder not found: {AREA_5_DIR}"

print(f"S3DIS path: {S3DIS_DIR}")
print(f"Area 5 path: {AREA_5_DIR}\n")

# ============================================================
# List all rooms in Area_5
# ============================================================

rooms = sorted([
    d for d in os.listdir(AREA_5_DIR)
    if os.path.isdir(os.path.join(AREA_5_DIR, d))
    and not d.startswith(".")
])

print(f"Rooms in Area_5 ({len(rooms)} total):\n")

for room in rooms:
    print(room)

S3DIS path: /Users/dosvatsky/3D Object Detection/data/S3DIS
Area 5 path: /Users/dosvatsky/3D Object Detection/data/S3DIS/Area_5

Rooms in Area_5 (68 total):

WC_1
WC_2
conferenceRoom_1
conferenceRoom_2
conferenceRoom_3
hallway_1
hallway_10
hallway_11
hallway_12
hallway_13
hallway_14
hallway_15
hallway_2
hallway_3
hallway_4
hallway_5
hallway_6
hallway_7
hallway_8
hallway_9
lobby_1
office_1
office_10
office_11
office_12
office_13
office_14
office_15
office_16
office_17
office_18
office_19
office_2
office_20
office_21
office_22
office_23
office_24
office_25
office_26
office_27
office_28
office_29
office_3
office_30
office_31
office_32
office_33
office_34
office_35
office_36
office_37
office_38
office_39
office_4
office_40
office_41
office_42
office_5
office_6
office_7
office_8
office_9
pantry_1
storage_1
storage_2
storage_3
storage_4


## Loading conferenceRoom_1

In [6]:
from collections import Counter
import os
import time
import numpy as np

# ============================================================
# Select room
# ============================================================

ROOM_NAME = "conferenceRoom_1"
room_dir = os.path.join(AREA_5_DIR, ROOM_NAME)

assert os.path.isdir(room_dir), (
    f"Room not found: {room_dir}\n"
    f"Available rooms:\n{sorted(os.listdir(AREA_5_DIR))[:20]}"
)

# ============================================================
# Load room point cloud
# ============================================================

print(f"Loading {ROOM_NAME}...")

t0 = time.time()
room_points = load_s3dis_room(room_dir)
load_time = time.time() - t0

print(f"  Whole-room points: {room_points.shape} ({room_points.shape[0]:,} points)")
print(
    f"  XYZ range: "
    f"x[{room_points[:,0].min():.2f}, {room_points[:,0].max():.2f}]  "
    f"y[{room_points[:,1].min():.2f}, {room_points[:,1].max():.2f}]  "
    f"z[{room_points[:,2].min():.2f}, {room_points[:,2].max():.2f}]"
)
print(f"  Load time: {load_time:.1f}s\n")

# ============================================================
# Load annotations
# ============================================================

print(f"Loading annotations for {ROOM_NAME}...")

t0 = time.time()
objects = load_s3dis_annotations(room_dir)
ann_time = time.time() - t0

print(f"  Found {len(objects)} annotated objects ({ann_time:.1f}s)\n")

# ============================================================
# Object statistics
# ============================================================

class_counts = Counter(obj["class_name"] for obj in objects)

print("Objects by class:\n")

for class_name, count in sorted(class_counts.items()):

    mapped_class = S3DIS_TO_MODELNET40.get(class_name, "—")

    avg_points = int(
        np.mean([
            obj["points"].shape[0]
            for obj in objects
            if obj["class_name"] == class_name
        ])
    )

    print(
        f"  {class_name:<12} "
        f"count: {count:>3}   "
        f"avg_points/object: {avg_points:>6,}   "
        f"maps to MN40 '{mapped_class}'"
    )

Loading conferenceRoom_1...
  Whole-room points: (1047554, 6) (1,047,554 points)
  XYZ range: x[-17.41, -10.64]  y[-5.92, -2.41]  z[-0.01, 3.63]
  Load time: 0.3s

Loading annotations for conferenceRoom_1...
  Found 29 annotated objects (0.3s)

Objects by class:

  board        count:   1   avg_points/object: 23,363   maps to MN40 '—'
  ceiling      count:   1   avg_points/object: 197,047   maps to MN40 '—'
  chair        count:   9   avg_points/object:  3,561   maps to MN40 'chair'
  clutter      count:   8   avg_points/object:  4,076   maps to MN40 '—'
  column       count:   1   avg_points/object: 42,130   maps to MN40 '—'
  door         count:   1   avg_points/object: 27,019   maps to MN40 'door'
  floor        count:   1   avg_points/object: 159,514   maps to MN40 '—'
  table        count:   2   avg_points/object: 15,791   maps to MN40 'table'
  wall         count:   4   avg_points/object: 86,502   maps to MN40 '—'
  window       count:   1   avg_points/object: 156,218   maps to M

## Visualizing Room withh Plotly

In [8]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# Downsample for visualization
# ============================================================

N_VIZ = 80_000

if room_points.shape[0] > N_VIZ:
    idx = np.random.choice(room_points.shape[0], N_VIZ, replace=False)
    viz_points = room_points[idx]
else:
    viz_points = room_points

print(f"Visualizing {viz_points.shape[0]:,} points")

# ============================================================
# RGB colors
# Assumes columns:
# [x, y, z, r, g, b]
# ============================================================

rgb = np.clip(viz_points[:, 3:6], 0, 255).astype(np.uint8)

colors = [
    f"rgb({r},{g},{b})"
    for r, g, b in rgb
]

# ============================================================
# Plotly 3D Scatter
# ============================================================

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=viz_points[:, 0],
            y=viz_points[:, 1],
            z=viz_points[:, 2],
            mode="markers",
            marker=dict(
                size=1.5,
                color=colors,
                opacity=0.85,
            ),
            hovertemplate=(
                "x=%{x:.2f}<br>"
                "y=%{y:.2f}<br>"
                "z=%{z:.2f}"
                "<extra></extra>"
            ),
        )
    ]
)

fig.update_layout(
    title=(
        f"S3DIS {ROOM_NAME} "
        f"({viz_points.shape[0]:,} points, RGB-colored)"
    ),
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data",
    ),
    width=900,
    height=700,
    margin=dict(
        l=0,
        r=0,
        t=40,
        b=0,
    ),
)

fig.show()

Visualizing 80,000 points
